# SCD TYPE-1 (Overwrite old value (no history))

In [0]:
%sql
CREATE TABLE datamodeling.default.scdtyp1_source
(
    prod_id INT,
    prod_name STRING,
    prod_category STRING,
    processDate DATE
)

In [0]:
%sql
INSERT INTO datamodeling.default.scdtyp1_source
VALUES  
(1, 'prod1', 'category1', current_date()),
(2, 'prod2', 'category2', current_date())

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
CREATE TABLE datamodeling.gold.scdtyp1_table
(
    prod_id INT,
    prod_name STRING,
    prod_category STRING,
    processDate DATE
)

In [0]:
# CREATE THE SOURCE
spark.sql("SELECT * FROM datamodeling.default.scdtyp1_source").createOrReplaceTempView("src")

In [0]:
%sql
-- UPSERT COMMAND
MERGE INTO datamodeling.gold.scdtyp1_table AS trg   -- trg = target
USING src
ON src.prod_id = trg.prod_id
WHEN MATCHED AND src.processDate >= trg.processDate THEN UPDATE SET *            -- Back Dated Refresh (update the value with the latest data)
WHEN NOT MATCHED THEN INSERT *


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


In [0]:
%sql

SELECT * FROM datamodeling.gold.scdtyp1_table

prod_id,prod_name,prod_category,processDate
3,prod3,new_category,2026-07-27
2,prod2,category2,2026-07-27
1,prod1,new_category_1,2026-07-27
2,prod2,category2,2026-07-27
1,prod1,new_category_1,2026-07-27


In [0]:
%sql
--UPDATE COMMAND
UPDATE datamodeling.default.scdtyp1_source
SET prod_category = "new_category_1"
WHERE prod_id = 1


num_affected_rows
1


# SCD TYPE-2 (Create a new row (full history))

In [0]:
%sql

CREATE TABLE datamodeling.default.scdtyp2_source
(   
    prod_id INT,
    prod_name STRING,
    prod_category STRING,
    processDate DATE
) 

In [0]:
%sql
INSERT INTO datamodeling.default.scdtyp2_source
VALUES
(1, 'prod1', 'new_category_1', current_date()),
(2, 'prod2', 'new_category_2', current_date()),
(3, 'prod3', 'new_category_3', current_date())

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
CREATE TABLE datamodeling.gold.scdtyp2_table
(
    prod_id INT,
    prod_name STRING,
    prod_category STRING,
    processDate DATE,
    start_date DATE,
    end_date DATE,
    is_current STRING
)

In [0]:
spark.sql("""SELECT 
                *,
                current_timestamp AS start_date,
                CAST('3000-01-01' AS timestamp) AS end_date,
                'Y' AS is_current    
          FROM datamodeling.default.scdtyp2_source""").createOrReplaceTempView("srctyp2")

In [0]:
%sql
SELECT * FROM srctyp2

prod_id,prod_name,prod_category,processDate,start_date,end_date,is_current
1,prod1,new_category_1,2026-07-29,2026-07-29T14:06:59.291Z,3000-01-01T00:00:00.000Z,Y
2,prod2,new_category_2,2026-07-29,2026-07-29T14:06:59.291Z,3000-01-01T00:00:00.000Z,Y
3,prod3,new_category_3,2026-07-29,2026-07-29T14:06:59.291Z,3000-01-01T00:00:00.000Z,Y


In [0]:
%sql
MERGE INTO datamodeling.gold.scdtyp2_table AS trg
USING srctyp2
ON srctyp2.prod_id = trg.prod_id
AND trg.is_current = 'Y'

-- When We Have New Data With Updates
WHEN MATCHED AND (                      -- when matched the prod_id but the other attributes are not, then inactivate the old row, write the new row
    srctyp2.prod_category <> trg.prod_category OR     
    srctyp2.prod_name <> trg.prod_name OR
    srctyp2.processDate <> trg.processDate
) THEN UPDATE SET
    trg.end_date = current_timestamp(),
    trg.is_current = "N"

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,1,0,0


In [0]:
%sql
-- Tách riêng ra vì nếu chạy cùng cell thì sau khi satisfy condition 1, sẽ dừng luôn
MERGE INTO datamodeling.gold.scdtyp2_table AS trg
USING srctyp2
ON srctyp2.prod_id = trg.prod_id
AND trg.is_current = 'Y'

-- When There Is New Data To Be Inserted
WHEN NOT MATCHED THEN INSERT
(
    prod_id,
    prod_name,
    prod_category,
    processDate,
    start_date,
    end_date,
    is_current
)
VALUES (
    srctyp2.prod_id,
    srctyp2.prod_name,
    srctyp2.prod_category,
    srctyp2.processDate,
    srctyp2.start_date,
    srctyp2.end_date,
    srctyp2.is_current
) 


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,0,0,1


In [0]:
%sql
SELECT * FROM datamodeling.gold.scdtyp2_table

prod_id,prod_name,prod_category,processDate,start_date,end_date,is_current
2,prod2,new_category_2,2026-07-29,2026-07-29,3000-01-01,Y
3,prod3,new_category_3,2026-07-29,2026-07-29,3000-01-01,Y
1,prod1,new_category_1,2026-07-29,2026-07-29,2026-07-29,N
1,prod1,test_cat1,2026-07-29,2026-07-29,3000-01-01,Y


In [0]:
%sql
UPDATE datamodeling.default.scdtyp2_source
SET prod_category = "test_cat1"
WHERE prod_id = 1

num_affected_rows
1


In [0]:
%sql
SELECT * FROM datamodeling.default.scdtyp2_source

prod_id,prod_name,prod_category,processDate
2,prod2,new_category_2,2026-07-29
3,prod3,new_category_3,2026-07-29
1,prod1,test_cat1,2026-07-29
